# Learning Rate Schedulers

**Interview Focus**: Scheduler implementations, warmup, cosine decay, standard LLM schedules.

**Key Concepts**:
- StepLR, CosineAnnealing, Linear Warmup, CyclicLR, OneCycleLR
- Warmup + Cosine Decay (the standard LLM schedule)
- Why warmup? Why cosine decay?
- Typical schedules for different tasks

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
import math

## 1. From-Scratch Implementations

In [ ]:
class StepLRScheduler:
    """Decay LR by gamma every step_size epochs."""
    def __init__(self, optimizer, step_size=30, gamma=0.1):
        self.optimizer = optimizer
        self.step_size = step_size
        self.gamma = gamma
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.current_step = 0
    
    def step(self):
        self.current_step += 1
        factor = self.gamma ** (self.current_step // self.step_size)
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = base_lr * factor
    
    def get_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]


class CosineAnnealingScheduler:
    """Cosine annealing from max_lr to min_lr over T_max steps.
    
    lr(t) = min_lr + 0.5 * (max_lr - min_lr) * (1 + cos(pi * t / T_max))
    """
    def __init__(self, optimizer, T_max, eta_min=0):
        self.optimizer = optimizer
        self.T_max = T_max
        self.eta_min = eta_min
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.current_step = 0
    
    def step(self):
        self.current_step += 1
        t = min(self.current_step, self.T_max)
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = self.eta_min + 0.5 * (base_lr - self.eta_min) * (
                1 + math.cos(math.pi * t / self.T_max)
            )
    
    def get_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]


class LinearWarmupScheduler:
    """Linear warmup from 0 to base_lr over warmup_steps."""
    def __init__(self, optimizer, warmup_steps):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.current_step = 0
    
    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            factor = self.current_step / self.warmup_steps
        else:
            factor = 1.0
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = base_lr * factor
    
    def get_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]


class WarmupCosineDecayScheduler:
    """Warmup + Cosine Decay — THE standard LLM training schedule.
    
    Phase 1: Linear warmup from 0 to max_lr (warmup_steps)
    Phase 2: Cosine decay from max_lr to min_lr (remaining steps)
    """
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=0):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.current_step = 0
    
    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            # Linear warmup
            factor = self.current_step / self.warmup_steps
            for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
                pg['lr'] = base_lr * factor
        else:
            # Cosine decay
            progress = (self.current_step - self.warmup_steps) / (
                self.total_steps - self.warmup_steps
            )
            progress = min(progress, 1.0)
            for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
                pg['lr'] = self.min_lr + 0.5 * (base_lr - self.min_lr) * (
                    1 + math.cos(math.pi * progress)
                )
    
    def get_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]


class CyclicLRScheduler:
    """Cyclic LR: triangular policy between base_lr and max_lr."""
    def __init__(self, optimizer, base_lr, max_lr, step_size_up=2000):
        self.optimizer = optimizer
        self.base_lr = base_lr
        self.max_lr = max_lr
        self.step_size_up = step_size_up
        self.step_size_down = step_size_up  # symmetric
        self.cycle_size = step_size_up + self.step_size_down
        self.current_step = 0
    
    def step(self):
        self.current_step += 1
        cycle = self.current_step % self.cycle_size
        if cycle < self.step_size_up:
            # Going up
            progress = cycle / self.step_size_up
        else:
            # Going down
            progress = 1.0 - (cycle - self.step_size_up) / self.step_size_down
        
        lr = self.base_lr + (self.max_lr - self.base_lr) * progress
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
    
    def get_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]


class OneCycleLRScheduler:
    """1Cycle: warmup to max_lr, then cosine decay to near zero.
    
    Phase 1 (pct_start): linear warmup from init_lr to max_lr
    Phase 2 (1-pct_start): cosine anneal from max_lr to min_lr
    """
    def __init__(self, optimizer, max_lr, total_steps, pct_start=0.3,
                 div_factor=25, final_div_factor=1e4):
        self.optimizer = optimizer
        self.max_lr = max_lr
        self.total_steps = total_steps
        self.pct_start = pct_start
        self.init_lr = max_lr / div_factor
        self.min_lr = self.init_lr / final_div_factor
        self.warmup_steps = int(total_steps * pct_start)
        self.current_step = 0
    
    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            # Phase 1: linear warmup
            progress = self.current_step / self.warmup_steps
            lr = self.init_lr + (self.max_lr - self.init_lr) * progress
        else:
            # Phase 2: cosine decay
            progress = (self.current_step - self.warmup_steps) / (
                self.total_steps - self.warmup_steps
            )
            lr = self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (
                1 + math.cos(math.pi * progress)
            )
        
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr
    
    def get_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]


print("All schedulers implemented from scratch.")

## 2. Visualize All Schedules

In [ ]:
def collect_lrs(scheduler_fn, total_steps):
    """Run a scheduler for total_steps and collect LR values."""
    # Dummy model and optimizer
    model = nn.Linear(10, 10)
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
    scheduler = scheduler_fn(optimizer)
    
    lrs = []
    for _ in range(total_steps):
        lrs.append(scheduler.get_lr()[0])
        scheduler.step()
    return lrs

total = 1000

schedules = {
    'StepLR (step=200, gamma=0.5)': lambda opt: StepLRScheduler(opt, step_size=200, gamma=0.5),
    'CosineAnnealing': lambda opt: CosineAnnealingScheduler(opt, T_max=total, eta_min=1e-5),
    'Linear Warmup (100 steps)': lambda opt: LinearWarmupScheduler(opt, warmup_steps=100),
    'Warmup + Cosine (LLM standard)': lambda opt: WarmupCosineDecayScheduler(
        opt, warmup_steps=100, total_steps=total, min_lr=1e-5),
    'CyclicLR': lambda opt: CyclicLRScheduler(opt, base_lr=1e-4, max_lr=1e-3, step_size_up=250),
    'OneCycleLR': lambda opt: OneCycleLRScheduler(opt, max_lr=1e-3, total_steps=total),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']

for idx, (name, scheduler_fn) in enumerate(schedules.items()):
    lrs = collect_lrs(scheduler_fn, total)
    axes[idx].plot(lrs, color=colors[idx], linewidth=2)
    axes[idx].set_title(name, fontsize=10)
    axes[idx].set_xlabel('Step')
    axes[idx].set_ylabel('Learning Rate')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].ticklabel_format(style='sci', axis='y', scilimits=(0,0))

plt.suptitle('Learning Rate Schedules (max_lr = 1e-3, 1000 steps)', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# All schedules on one plot for comparison
fig, ax = plt.subplots(figsize=(12, 5))

for (name, scheduler_fn), color in zip(schedules.items(), colors):
    lrs = collect_lrs(scheduler_fn, total)
    ax.plot(lrs, label=name, color=color, linewidth=2)

ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('All Schedules Compared')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
plt.tight_layout()
plt.show()

## 3. Training Comparison

In [ ]:
# Create dataset
torch.manual_seed(42)
n = 2000
X = torch.randn(n, 20)
w_true = torch.randn(20, 1) * 0.5
y = (X @ w_true + 0.1 * torch.randn(n, 1)).squeeze()
y_cls = (y > y.median()).long()

dataset = TensorDataset(X, y_cls)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

steps_per_epoch = len(train_loader)
n_epochs = 30
total_steps = steps_per_epoch * n_epochs

def make_model():
    torch.manual_seed(0)
    return nn.Sequential(
        nn.Linear(20, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 2)
    )

def train_with_scheduler(scheduler_fn, name):
    model = make_model()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
    scheduler = scheduler_fn(optimizer)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    accs = []
    
    for epoch in range(n_epochs):
        epoch_loss = 0
        correct = 0
        total = 0
        
        for inputs, targets in train_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            epoch_loss += loss.item()
            correct += (outputs.argmax(1) == targets).sum().item()
            total += len(targets)
        
        losses.append(epoch_loss / steps_per_epoch)
        accs.append(correct / total)
    
    return losses, accs


# Train with different schedulers
results = {}

warmup_steps = steps_per_epoch * 3  # 3 epochs warmup

scheduler_configs = {
    'Constant LR': lambda opt: type('Const', (), {'step': lambda self: None, 'get_lr': lambda self: [opt.param_groups[0]['lr']]})(),
    'StepLR': lambda opt: StepLRScheduler(opt, step_size=steps_per_epoch * 10, gamma=0.1),
    'CosineAnnealing': lambda opt: CosineAnnealingScheduler(opt, T_max=total_steps, eta_min=1e-4),
    'Warmup+Cosine': lambda opt: WarmupCosineDecayScheduler(
        opt, warmup_steps=warmup_steps, total_steps=total_steps, min_lr=1e-4),
    'OneCycle': lambda opt: OneCycleLRScheduler(opt, max_lr=0.05, total_steps=total_steps),
}

for name, sched_fn in scheduler_configs.items():
    losses, accs = train_with_scheduler(sched_fn, name)
    results[name] = (losses, accs)
    print(f"{name:>20}: final_loss={losses[-1]:.4f}, final_acc={accs[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, (losses, accs) in results.items():
    axes[0].plot(losses, label=name, linewidth=2)
    axes[1].plot(accs, label=name, linewidth=2)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training with Different LR Schedulers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. The Standard LLM Schedule: Warmup + Cosine Decay

This is what GPT-3, LLaMA, and most large language models use.

In [ ]:
# LLaMA-style schedule
total_tokens = 1.4e12  # 1.4T tokens
batch_tokens = 4e6     # 4M tokens per batch (batch_size * seq_len)
total_steps_llm = int(total_tokens / batch_tokens)  # 350K steps
warmup_steps_llm = 2000
max_lr_llm = 3e-4
min_lr_llm = max_lr_llm * 0.1  # common to decay to 10% of peak

print(f"LLaMA-style training schedule:")
print(f"  Total steps:   {total_steps_llm:,}")
print(f"  Warmup steps:  {warmup_steps_llm:,}")
print(f"  Peak LR:       {max_lr_llm}")
print(f"  Min LR:        {min_lr_llm}")

# Simulate the schedule
steps = np.arange(total_steps_llm)
lrs_llm = []
for s in steps:
    if s < warmup_steps_llm:
        lr = max_lr_llm * (s / warmup_steps_llm)
    else:
        progress = (s - warmup_steps_llm) / (total_steps_llm - warmup_steps_llm)
        lr = min_lr_llm + 0.5 * (max_lr_llm - min_lr_llm) * (1 + math.cos(math.pi * progress))
    lrs_llm.append(lr)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(steps, lrs_llm, 'b-', linewidth=2)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Learning Rate')
axes[0].set_title('LLaMA-Style Schedule (Full)')
axes[0].axvline(x=warmup_steps_llm, color='red', linestyle='--', alpha=0.5, label='End warmup')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Zoom into warmup
zoom = 5000
axes[1].plot(steps[:zoom], lrs_llm[:zoom], 'b-', linewidth=2)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Warmup Phase (Zoomed)')
axes[1].axvline(x=warmup_steps_llm, color='red', linestyle='--', alpha=0.5, label='End warmup')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Using PyTorch Built-in Schedulers

In [ ]:
from torch.optim.lr_scheduler import (
    StepLR, CosineAnnealingLR, CyclicLR, OneCycleLR,
    LambdaLR, SequentialLR, LinearLR
)

# Warmup + Cosine using LambdaLR (most flexible)
def get_warmup_cosine_schedule(optimizer, warmup_steps, total_steps, min_lr_ratio=0.1):
    """Create warmup + cosine decay schedule using LambdaLR."""
    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return min_lr_ratio + 0.5 * (1.0 - min_lr_ratio) * (1 + math.cos(math.pi * progress))
    
    return LambdaLR(optimizer, lr_lambda)


# Demo
model = nn.Linear(10, 10)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
scheduler = get_warmup_cosine_schedule(optimizer, warmup_steps=100, total_steps=1000)

lrs = []
for step in range(1000):
    lrs.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

plt.figure(figsize=(10, 4))
plt.plot(lrs, linewidth=2)
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.title('Warmup + Cosine Decay (PyTorch LambdaLR)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Typical Schedules by Task

| Task | Typical Schedule | Peak LR | Notes |
|------|-----------------|---------|-------|
| **LLM pretraining** | Warmup + Cosine | 1e-4 to 3e-4 | Warmup 0.1-1% of steps |
| **LLM fine-tuning** | Warmup + Linear decay | 1e-5 to 5e-5 | Short warmup (6% of steps) |
| **BERT** | Warmup + Linear decay | 1e-4 | 10K warmup steps |
| **Vision (ResNet)** | StepLR or CosineAnnealing | 0.1 (SGD) | Decay at 30, 60, 90 epochs |
| **Vision (ViT)** | Warmup + Cosine | 1e-3 | 10K warmup steps |
| **Super-convergence** | OneCycleLR | high | Best for fast training |

In [ ]:
# Show typical schedules side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# LLM Pretraining
steps = 10000
warmup = 500
lrs = []
for s in range(steps):
    if s < warmup:
        lr = 3e-4 * s / warmup
    else:
        p = (s - warmup) / (steps - warmup)
        lr = 3e-5 + 0.5 * (3e-4 - 3e-5) * (1 + math.cos(math.pi * p))
    lrs.append(lr)
axes[0].plot(lrs, 'b-', linewidth=2)
axes[0].set_title('LLM Pretraining\n(Warmup + Cosine)')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('LR')
axes[0].grid(True, alpha=0.3)

# BERT Fine-tuning
steps = 3000
warmup = 180  # 6%
lrs = []
for s in range(steps):
    if s < warmup:
        lr = 2e-5 * s / warmup
    else:
        lr = 2e-5 * (1 - (s - warmup) / (steps - warmup))
    lrs.append(lr)
axes[1].plot(lrs, 'r-', linewidth=2)
axes[1].set_title('BERT Fine-tuning\n(Warmup + Linear Decay)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('LR')
axes[1].grid(True, alpha=0.3)

# Vision (ResNet)
epochs = 90
lrs = []
lr = 0.1
for e in range(epochs):
    if e == 30:
        lr *= 0.1
    elif e == 60:
        lr *= 0.1
    lrs.append(lr)
axes[2].plot(lrs, 'g-', linewidth=2)
axes[2].set_title('ResNet Training\n(StepLR @ 30, 60)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('LR')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Interview Questions & Answers

---

**Q: Why warmup?**

At initialization, model weights are random and gradients have high variance. A large LR would cause destructive updates. Warmup gradually increases LR so:
1. Early updates are small and stable
2. Adam's running statistics (mean, variance) warm up to reasonable values
3. The model enters a stable optimization region before using the full LR

Particularly important for: transformers, large batch sizes, and adaptive optimizers.

---

**Q: Why cosine decay?**

1. Smooth decay avoids sudden drops (gentler than StepLR)
2. Spends most time near the minimum LR (fine refinement)
3. Empirically works well for LLM pretraining — many papers confirm this
4. The slow initial decay allows continued exploration, fast final decay encourages convergence

---

**Q: How to set the max learning rate?**

1. **LR range test** (Leslie Smith): train for a few hundred steps with exponentially increasing LR. Pick the LR just before loss starts diverging.
2. **Rules of thumb**: Adam ~1e-4 to 3e-4, SGD ~0.01 to 0.1
3. **Linear scaling rule**: If you double batch size, double LR (but add warmup)
4. **Established baselines**: Use what worked in similar published work

---

**Q: Scheduler per step or per epoch?**

- **Per step**: Warmup, CosineAnnealing, OneCycle — most modern schedulers
- **Per epoch**: StepLR, ReduceLROnPlateau — older convention
- LLM training: **always per step** (one epoch may be millions of steps)

## Quick Reference

```python
# Standard LLM schedule
from torch.optim.lr_scheduler import LambdaLR

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.1 + 0.9 * 0.5 * (1 + cos(pi * progress))  # decay to 10%

scheduler = LambdaLR(optimizer, lr_lambda)

# Call scheduler.step() AFTER optimizer.step() on every training step
```